# ARIMA 模型分析 - 虛擬貨幣價格預測

本筆記本展示如何使用 ARIMA (AutoRegressive Integrated Moving Average) 模型來預測虛擬貨幣價格。

## 學習目標
1. 理解 ARIMA 模型的基本概念
2. 學會檢測時間序列的平穩性
3. 掌握 ARIMA 參數選擇方法
4. 實作價格預測並評估模型效果

## ARIMA 模型簡介

ARIMA(p,d,q) 模型由三個部分組成：
- **AR(p)**: 自回歸項，使用過去 p 期的觀測值
- **I(d)**: 差分項，進行 d 次差分使序列平穩
- **MA(q)**: 移動平均項，使用過去 q 期的預測誤差

In [ ]:
# 導入必要的套件
import sys
import os
sys.path.append('../../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# 設定圖表樣式
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# 導入自定義 ARIMA 模型
from models.foundations.arima_model import ARIMAModel

print("套件導入完成！")

## 1. 數據準備

首先我們需要準備虛擬貨幣的歷史價格數據。在實際應用中，這些數據會從交易所 API 獲取。

In [ ]:
# 生成模擬的 BTC 價格數據
# 在實際應用中，這裡會連接到 Binance 或 Bybit API

def generate_crypto_price_data(start_price=30000, days=365, freq='1H'):
    """
    生成模擬的加密貨幣價格數據
    """
    np.random.seed(42)
    
    # 創建時間索引
    if freq == '1H':
        periods = days * 24
    elif freq == '1D':
        periods = days
    else:
        periods = days * 24
    
    dates = pd.date_range(start='2023-01-01', periods=periods, freq=freq)
    
    # 模擬價格走勢（含趨勢、季節性和隨機波動）
    trend = np.linspace(0, 0.3, periods)  # 上升趨勢
    seasonal = 0.1 * np.sin(2 * np.pi * np.arange(periods) / (24 * 7))  # 週期性
    noise = np.random.normal(0, 0.02, periods)  # 隨機波動
    
    # 組合成價格序列
    log_returns = trend / periods + seasonal + noise
    
    prices = [start_price]
    for ret in log_returns:
        prices.append(prices[-1] * np.exp(ret))
    
    # 創建 OHLCV 數據
    close_prices = np.array(prices[1:])
    high_prices = close_prices * (1 + np.abs(np.random.normal(0, 0.01, len(close_prices))))
    low_prices = close_prices * (1 - np.abs(np.random.normal(0, 0.01, len(close_prices))))
    open_prices = np.roll(close_prices, 1)
    open_prices[0] = start_price
    volumes = np.random.lognormal(10, 1, len(close_prices))
    
    data = pd.DataFrame({
        'open': open_prices,
        'high': high_prices,
        'low': low_prices,
        'close': close_prices,
        'volume': volumes
    }, index=dates)
    
    return data

# 生成 BTC 小時級數據
btc_data = generate_crypto_price_data(start_price=30000, days=90, freq='1H')

print(f"數據期間: {btc_data.index[0]} 到 {btc_data.index[-1]}")
print(f"數據點數: {len(btc_data)}")
print("\n數據預覽:")
btc_data.head()

In [ ]:
# 繪製價格走勢圖
plt.figure(figsize=(15, 8))

plt.subplot(2, 1, 1)
plt.plot(btc_data.index, btc_data['close'], linewidth=1)
plt.title('BTC 收盤價格走勢', fontsize=14, fontweight='bold')
plt.ylabel('價格 (USD)')
plt.grid(True, alpha=0.3)

plt.subplot(2, 1, 2)
returns = btc_data['close'].pct_change().dropna()
plt.plot(returns.index, returns, linewidth=0.8, alpha=0.7)
plt.title('BTC 收益率', fontsize=14, fontweight='bold')
plt.ylabel('收益率')
plt.xlabel('時間')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 基礎統計信息
print("=== BTC 價格統計信息 ===")
print(btc_data['close'].describe())

## 2. ARIMA 模型分析

現在我們使用自定義的 ARIMAModel 類來進行完整的分析流程。

In [ ]:
# 初始化 ARIMA 模型
arima_model = ARIMAModel(btc_data, price_column='close')

print("ARIMA 模型初始化完成")
print(f"數據期間: {btc_data.index[0]} 到 {btc_data.index[-1]}")
print(f"價格範圍: ${btc_data['close'].min():.2f} - ${btc_data['close'].max():.2f}")

### 2.1 平穩性檢測

ARIMA 模型要求時間序列是平穩的。我們使用 ADF 檢定來檢測平穩性。

In [ ]:
# 檢測原始價格序列的平穩性
print("=== 原始價格序列平穩性檢測 ===")
stationarity_result = arima_model.check_stationarity()

if not stationarity_result['is_stationary']:
    print("\n原始序列非平穩，需要進行差分處理")
    
    # 進行一階差分
    print("\n=== 一階差分後的平穩性檢測 ===")
    diff_series = arima_model.make_stationary(method='difference')
else:
    print("\n原始序列已經平穩，可直接建模")

### 2.2 ACF 和 PACF 分析

通過分析自相關函數 (ACF) 和偏自相關函數 (PACF) 來初步確定 ARIMA 參數。

In [ ]:
# 繪製 ACF 和 PACF 圖
print("=== ACF 和 PACF 分析 ===")

# 對差分後的序列進行分析
diff_series = btc_data['close'].diff().dropna()
arima_model.plot_acf_pacf(series=diff_series, lags=48)  # 48小時的滯後

print("\n參數選擇指導:")
print("- AR(p): 觀察 PACF 圖，在第 p 階後截斷")
print("- MA(q): 觀察 ACF 圖，在第 q 階後截斷")
print("- 如果序列經過一次差分變平穩，則 d=1")

### 2.3 自動參數選擇

使用網格搜索和信息準則 (AIC) 來自動選擇最優的 ARIMA 參數。

In [ ]:
# 自動尋找最佳 ARIMA 參數
print("=== 自動參數選擇 ===")
print("正在搜索最優參數...（這可能需要幾分鐘）")

best_params, results_df = arima_model.find_best_arima_params(
    p_range=range(0, 4),
    d_range=range(0, 3),
    q_range=range(0, 4),
    ic='aic'
)

print(f"\n最佳參數組合: ARIMA{best_params}")

# 可視化參數搜索結果
if len(results_df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    for i, ic in enumerate(['aic', 'bic', 'hqic']):
        top_models = results_df.nsmallest(10, ic)
        axes[i].bar(range(len(top_models)), top_models[ic])
        axes[i].set_title(f'Top 10 Models by {ic.upper()}')
        axes[i].set_xlabel('Model Rank')
        axes[i].set_ylabel(ic.upper())
        
        # 標註參數
        for j, (idx, row) in enumerate(top_models.iterrows()):
            axes[i].text(j, row[ic], f'({int(row.p)},{int(row.d)},{int(row.q)})', 
                        ha='center', va='bottom', fontsize=8, rotation=45)
    
    plt.tight_layout()
    plt.show()

### 2.4 模型擬合

使用選出的最佳參數來擬合 ARIMA 模型。

In [ ]:
# 擬合 ARIMA 模型
print("=== 模型擬合 ===")
fitted_model = arima_model.fit_arima(order=best_params)

print("\n模型擬合完成！")
print(f"模型參數: ARIMA{best_params}")
print(f"AIC: {fitted_model.aic:.2f}")
print(f"BIC: {fitted_model.bic:.2f}")
print(f"Log Likelihood: {fitted_model.llf:.2f}")

### 2.5 模型診斷

檢查模型殘差的特性，確保模型適合度。

In [ ]:
# 模型診斷
print("=== 模型診斷 ===")
arima_model.model_diagnostics()

print("\n診斷要點:")
print("1. 殘差圖: 殘差應該圍繞零隨機波動，無明顯模式")
print("2. Q-Q圖: 點應該沿著直線分布，表示殘差符合正態分布")
print("3. 殘差ACF: 大部分滯後項應在置信區間內，表示無自相關")
print("4. 殘差直方圖: 應近似正態分布")

### 2.6 價格預測

使用擬合好的模型進行未來價格預測。

In [ ]:
# 生成預測
print("=== 價格預測 ===")

# 預測未來24小時的價格
forecast_steps = 24
forecast_result = arima_model.forecast(steps=forecast_steps, confidence_level=0.95)

print(f"預測期間: {forecast_result['forecast'].index[0]} 到 {forecast_result['forecast'].index[-1]}")
print(f"當前價格: ${btc_data['close'].iloc[-1]:.2f}")
print(f"24小時後預測價格: ${forecast_result['forecast'].iloc[-1]:.2f}")

# 詳細預測結果
forecast_df = pd.DataFrame({
    '預測價格': forecast_result['forecast'],
    '下界 (95%)': forecast_result['lower_ci'],
    '上界 (95%)': forecast_result['upper_ci']
})

print("\n未來24小時預測 (每6小時):")
print(forecast_df.iloc[::6].round(2))  # 每6小時顯示一次

In [ ]:
# 繪製預測結果
print("=== 預測結果視覺化 ===")
arima_model.plot_forecast(train_periods=200)  # 顯示最近200個時點的歷史數據

# 計算預測變化率
current_price = btc_data['close'].iloc[-1]
predicted_price_24h = forecast_result['forecast'].iloc[-1]
change_pct = ((predicted_price_24h - current_price) / current_price) * 100

print(f"\n24小時價格變化預測: {change_pct:+.2f}%")
if change_pct > 0:
    print("模型預測價格將上漲 📈")
else:
    print("模型預測價格將下跌 📉")

### 2.7 模型評估

評估模型的預測準確性和可靠性。

In [ ]:
# 模型評估
print("=== 模型評估 ===")
evaluation_metrics = arima_model.evaluate_model()

# 計算額外的評估指標
fitted_values = fitted_model.fittedvalues
actual_values = btc_data['close'].loc[fitted_values.index]

# 方向準確率 (預測漲跌方向的準確性)
actual_direction = np.sign(actual_values.diff()).dropna()
predicted_direction = np.sign(fitted_values.diff()).dropna()

# 對齊索引
common_index = actual_direction.index.intersection(predicted_direction.index)
direction_accuracy = (actual_direction[common_index] == predicted_direction[common_index]).mean()

print(f"\n額外評估指標:")
print(f"方向準確率: {direction_accuracy:.2%}")
print(f"價格解釋變異度 (R²): {fitted_model.rsquared:.4f}")

# 視覺化擬合效果
plt.figure(figsize=(15, 8))

plt.subplot(2, 1, 1)
recent_data = btc_data['close'].tail(200)
recent_fitted = fitted_values.loc[fitted_values.index.intersection(recent_data.index)]

plt.plot(recent_data.index, recent_data.values, label='實際價格', alpha=0.8)
plt.plot(recent_fitted.index, recent_fitted.values, label='擬合價格', alpha=0.8)
plt.title('ARIMA 模型擬合效果 (最近200個時點)')
plt.ylabel('價格 (USD)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 1, 2)
residuals = actual_values - fitted_values
recent_residuals = residuals.tail(200)
plt.plot(recent_residuals.index, recent_residuals.values, alpha=0.7)
plt.axhline(y=0, color='red', linestyle='--', alpha=0.5)
plt.title('模型殘差 (最近200個時點)')
plt.ylabel('殘差 (USD)')
plt.xlabel('時間')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. 實戰應用建議

### 3.1 模型限制與改進方向

ARIMA 模型的限制：
- 假設線性關係，無法捕捉非線性模式
- 對異常值敏感
- 需要平穩時間序列
- 無法處理多變量信息

改進建議：
- 結合外部變量 (如交易量、市場情緒指標)
- 使用 ARIMAX 或 SARIMAX 模型
- 考慮非線性模型 (如 LSTM)
- 實施滾動窗口預測

In [ ]:
# 總結與下一步
print("=== ARIMA 模型分析總結 ===")
print(f"✅ 最佳模型: ARIMA{best_params}")
print(f"✅ 模型AIC: {fitted_model.aic:.2f}")
print(f"✅ 方向準確率: {direction_accuracy:.2%}")
print(f"✅ RMSE: {evaluation_metrics['rmse']:.2f} USD")
print(f"✅ MAPE: {evaluation_metrics['mape']:.2f}%")

print("\n🎯 下一步學習計劃:")
print("1. 學習 GARCH 模型來預測波動率")
print("2. 實作 GBM 模型來模擬價格路徑")
print("3. 結合多個模型提高預測準確性")
print("4. 開發實時數據接口連接交易所API")

print("\n📊 模型參數保存:")
model_params = {
    'best_order': best_params,
    'aic': fitted_model.aic,
    'bic': fitted_model.bic,
    'rmse': evaluation_metrics['rmse'],
    'mape': evaluation_metrics['mape'],
    'direction_accuracy': direction_accuracy
}

print(model_params)

## 4. 實作練習

嘗試以下練習來加深理解：

1. **不同時間頻率**: 嘗試使用日線或分鐘線數據，觀察模型表現差異
2. **滾動預測**: 實作滾動窗口預測，每次只使用固定期間的歷史數據
3. **多幣種比較**: 對不同加密貨幣 (ETH, BNB 等) 建立 ARIMA 模型並比較
4. **季節性調整**: 嘗試 SARIMA 模型來處理可能的季節性模式
5. **交易策略**: 基於 ARIMA 預測結果設計簡單的交易策略

**作業**: 使用真實的交易所數據 (可通過 Binance API) 重複本分析流程。